# Solving ODEs with PhysicsNeMo

[Full course sequence](../../ai4sci/README.md) | **3/9 · Projectile** | Previous: [PINN fundamentals](../introduction/Introductory_Notebook.ipynb) | Next: [Diffusion](../diffusion_1d/Diffusion_Problem_Notebook.ipynb)

This notebook targets **nvidia-physicsnemo==2.2.2**. It preserves the course problems and sequence while using the current API for models, physics residuals, losses, and training loops. A successful short run does not certify convergence or physical accuracy. Review the fixed evaluation errors in `metrics.json`, the training history in `loss.csv`, and the prediction plots together.

The `.ipynb` file provides explanations, execution cells, and result inspection; `.py` contains the actual training program; `.yaml` contains configuration. Read the examples, open the linked `.py` file in the JupyterLab editor, then **edit → save → rerun the execution cell**. Editing a Markdown code block does not change the program.

## Simulating the Projectile Motion ODE

Our aim in this setup is to obtain the position of a point mass as a function of time $ ( x(t) , y(t) )$ which is given an initial velocity from the origin at the initial moment ( $t=0$ ), this 2-d space has a constant gravitational acceleration ( $g=-9.81m/s^2$ ) and air resistance can be neglected. The sample output path of the problem can be seen below. While we are familiar with the equations of motions, we can differentiate it to obtain the differential equations for this system.
<center><img src="images/projectile.svg" alt="Drawing" style="width:600px" /></center>


The equations of motion for such a system are defined as follows: 
$$
\begin{align}
S_x &= V_{0x} t \\
S_y &= V_{0y} t + \frac{1}{2} g t^2 
\end{align}
$$
The above equation when differentiated with respect to time gives us the following:  
$$
\begin{align}
\frac{dS_x}{dt} &= V_{0x} \\
\frac{dS_y}{dt} &= V_{0y} + gt
\end{align}
$$
While you may setup the problem statement with these equations, in this setup we will be using further differentiate it to get the second-order derivatives with respect to time as following and using it in our setup: 
$$
\begin{align}
\frac{\mathrm{d}^2 S_x}{\mathrm{d} t^2} &= 0 \\
\frac{\mathrm{d}^2 S_y}{\mathrm{d} t^2} &= g
\end{align}
$$
The above second order differential equations can now be used to setup our case, in this case while we are aware of the analytical solutions, a lot of the complex systems are defined using Partial differential equations and finding an analytical solution might not be straight-forward when the complexity of equations increases.

### Step 1: Time domain and initial conditions

Use $t\in[0,5]$, $v_0=40$ m/s, $\theta=\pi/3$, $x(0)=y(0)=0$, $x_t(0)=20$, and $y_t(0)=40\sin(\pi/3)$. Sample time coordinates directly without introducing an artificial spatial point. Evaluate inference over $5<t\le8$ separately as extrapolation outside the training interval.

The gravity value in the original narrative is changed from -9.8 to -9.81 to match the original executable code.

### Step 2: Symbolic equations and model

```python
class ProjectileEquation(PDE):
    def __init__(self, gravity=9.81):
        self.dim = 1
        t = Symbol("t")
        x, y = Function("x")(t), Function("y")(t)
        self.equations = {"ode_x": x.diff(t, 2), "ode_y": y.diff(t, 2) + gravity}
```

```python
class ProjectileModel(torch.nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.network = mlp(1, 2, cfg)

    def forward(self, t):
        return 100.0 * self.network(2.0 * t / 5.0 - 1.0)
```

Setting `ode_y = y_tt + 9.81` to zero is equivalent to the original target `y_tt = -9.81`. In version 2.2.2, `PhysicsInformer` automatically differentiates spatial coordinates x/y/z, so compute time derivatives with `torch.autograd.grad` and supply `x__t__t` and `y__t__t` explicitly. Time remains a time coordinate.

### Step 3: Initial and ODE losses

```python
def loss_terms(model, physics, batch_size, device):
    t = (5.0 * torch.rand(batch_size, 1, device=device)).requires_grad_()
    res = residuals(model(t), t, physics)
    t0 = torch.zeros(batch_size, 1, device=device, requires_grad=True)
    xy0 = model(t0)
    vx, vy = derivative(xy0[:, :1], t0), derivative(xy0[:, 1:], t0)
    return {"physics": sum((v / 9.81).square().mean() for v in res.values()),
            "initial_position": (xy0 / 100).square().mean(),
            "initial_velocity": ((vx - 20) / 40).square().mean()
                                + ((vy - 40 * math.sin(math.pi / 3)) / 40).square().mean()}
```

The losses above evaluate initial position, initial velocity, and ODE residual separately. Normalization factors for the physical scales are explicit in the code. The output scale of 100 and the input-time normalization are numerical training choices; the physical units and equations are preserved.

### Step 4: Validation and inference

The analytical solution $x=20t$, $y=40\sin(\pi/3)t-9.81t^2/2$ is used only for evaluation. `heldout_before/after` reports solution and PDE errors at fixed interior coordinates. `in_domain_rmse` and `extrapolation_rmse` are saved separately.

### Step 5: Configuration

[config.yaml](source_code/conf/config.yaml) defines steps, batch_size, learning_rate, layer_size, and num_layers. These are teaching settings, not certified convergence settings or an official tuning recommendation.

```yaml
steps: 5000
batch_size: 128
learning_rate: 0.001
layer_size: 64
num_layers: 3
```

### Step 6: Explicit training loop

The PDE used by [projectile.py](source_code/projectile.py) is defined in [projectile_eqn.py](source_code/projectile_eqn.py). The loop follows `optimizer.zero_grad → loss.backward → optimizer.step`. Shared training and saving utilities are in [runtime.py](../runtime.py). After the first execution check, increase STEPS and compare convergence.

In [ ]:
import os, sys, json, subprocess, uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial").is_dir() and (p / "challenge").is_dir())
LAB = ROOT / "tutorial/projectile"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs")))
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # increase after the execution check
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
OUTPUT = OUTPUT_BASE / ("projectile-" + uuid.uuid4().hex[:8])
command = [sys.executable, str(LAB / "source_code/projectile.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)]
subprocess.run(command, check=True, cwd=ROOT)
print(json.loads((OUTPUT / "metrics.json").read_text()))

### Visualizing the solution

In [ ]:
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for j, name in enumerate(("x", "y")):
    axes[j].plot(data["t"], data["reference"][:, j], label="analytical")
    axes[j].plot(data["t"], data["prediction"][:, j], label="PINN")
    axes[j].axvline(5, color="grey", linestyle="--")
    axes[j].set(xlabel="time (s)", ylabel=name + " (m)"); axes[j].legend()
plt.show()

## Introduction to ParaView

ParaView is a tool for visualizing scientific data in 2D and 3D. Install the version for your operating system from the [official download page](https://www.paraview.org/download/). This version of the exercise saves `.npz` results by default. The next cell exports the same predictions to CSV. Open the CSV in ParaView and apply **Table To Points**, selecting X Column=x, Y Column=y, and Z Column=z. Color the trajectory by time t.

The VTP output paths and screenshots from the historical course do not describe this run. Use `predictions.npz` and the CSV exported below for the current results.

In [ ]:
xyz_t = np.column_stack((data["prediction"], np.zeros(len(data["t"])), data["t"]))
np.savetxt(OUTPUT / "trajectory.csv", xyz_t, delimiter=",", header="x,y,z,t", comments="")
print(OUTPUT / "trajectory.csv")

### Next steps

[Full course sequence](../../ai4sci/README.md) | **3/9 · Projectile** | Previous: [PINN fundamentals](../introduction/Introductory_Notebook.ipynb) | Next: [Diffusion](../diffusion_1d/Diffusion_Problem_Notebook.ipynb)

--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.